In [1]:
# =========================================
# NLP Task 3: Chatbot using Transformers
# =========================================

"""
Name: Kaushal Boharupi
Task: NLP Task 3 - Chatbot using Hugging Face Transformers

Description:
This chatbot uses DialoGPT (transformer model) to generate
human-like responses and maintain conversation flow.
"""

# Suppress warnings (clean output)
import warnings
warnings.filterwarnings("ignore")

# Import libraries
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# =========================================
# 1. Load Model & Tokenizer
# =========================================

model_name = "microsoft/DialoGPT-small"   # lightweight & fast

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Fix padding issue
tokenizer.pad_token = tokenizer.eos_token

# =========================================
# 2. Chatbot Function
# =========================================

def chatbot():
    print("🤖 Chatbot: Hello! I am your AI assistant. Type 'exit' to quit.\n")

    chat_history_ids = None

    while True:
        # Take user input
        user_input = input("You: ")

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("🤖 Chatbot: Goodbye! Have a great day! 👋")
            break

        # Encode input with attention mask
        new_input = tokenizer(user_input + tokenizer.eos_token, return_tensors='pt')
        input_ids = new_input['input_ids']
        attention_mask = new_input['attention_mask']

        # Maintain conversation history
        if chat_history_ids is not None:
            input_ids = torch.cat([chat_history_ids, input_ids], dim=-1)
            attention_mask = torch.ones_like(input_ids)

        # Generate response
        chat_history_ids = model.generate(
            input_ids,
            attention_mask=attention_mask,
            max_length=200,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=40,
            top_p=0.9,
            temperature=0.7
        )

        # Decode only new response
        response = tokenizer.decode(
            chat_history_ids[:, input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        print("🤖 Chatbot:", response)
        print()

# =========================================
# 3. Run Chatbot
# =========================================

chatbot()

Loading weights: 100%|██████████| 149/149 [00:00<00:00, 9203.45it/s]


🤖 Chatbot: Hello! I am your AI assistant. Type 'exit' to quit.

🤖 Chatbot: I'm on mobile, I can't see the username or the post.

🤖 Chatbot: I'm in.

🤖 Chatbot: Goodbye! Have a great day! 👋
